# 03. BI Star Schema Construction

**Goal:** Build a dimensional model (Star Schema) on top of the Business Intelligence dataset (`loans_gold_bi.parquet`).

> **Documentation Note (Why these dimensions?):**
> - **`dim_date`**: Created to pre-calculate standard Date hierarchies (Year, Quarter, Month). This prevents Power BI from generating heavy hidden Auto Date/Time tables in memory, keeping the `.pbix` file light and fast.
> - **`dim_risk`**: Decodes numeric categories (e.g., `mths_since_recent_inq_cat = 1`) into clean string labels (e.g., `'No Record'`) directly in SQL, avoiding complex DAX calculated columns in Power BI.
> - **`dim_purpose`**: Standard lookup table to filter loans by destination (e.g., 'small_business').
> - **`dim_geography`**: Maps 2-letter state codes (e.g., 'MS') to fully qualified names (e.g., 'Mississippi, USA') to ensure zero-ambiguity rendering in Azure Maps visual.
> - **`fact_loans`**: Retains *all* ~55 analytical columns from the Silver layer. Demographics remain here as degenerate dimensions since we lack a unique `customer_id`. A calculated column `annual_inc_category` is added here for immediate consumption in BI charts.

In [23]:
import duckdb
import pandas as pd
import os

os.makedirs('../data/gold', exist_ok=True)

# Connect to new DuckDB Database for BI
con = duckdb.connect('../data/gold/goldBI.duckdb')

# Load BI parquet data
con.execute("""
    CREATE OR REPLACE VIEW bi_data AS 
    SELECT * FROM read_parquet('../data/gold/loans_gold_bi.parquet')
""")

print("BI layer loaded ")
print(con.execute("SELECT COUNT(*) FROM bi_data").fetchone())

BI layer loaded 
(2257158,)


In [24]:
# ── dim_date ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY issue_d) AS date_key,
        issue_d::DATE AS issue_d,
        YEAR(issue_d::DATE) AS year,
        QUARTER(issue_d::DATE) AS quarter,
        MONTH(issue_d::DATE) AS month_num,
        STRFTIME(issue_d::DATE, '%b') AS month_name
    FROM (SELECT DISTINCT issue_d FROM bi_data)
    ORDER BY issue_d
""")

print("dim_date ")
print(con.execute("SELECT COUNT(*) FROM dim_date").fetchone())

dim_date 
(139,)


In [25]:
# ── dim_risk ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE dim_risk AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY sub_grade, term, mths_since_recent_inq_cat, mths_since_recent_bc_cat) AS risk_key,
        sub_grade,
        term,
        mths_since_recent_inq_cat,
        CASE 
            WHEN mths_since_recent_inq_cat = 0 THEN 'Pre-2012 (No Coverage)'
            WHEN mths_since_recent_inq_cat = 1 THEN 'No Record'
            WHEN mths_since_recent_inq_cat = 2 THEN '>=3 Years Ago'
            WHEN mths_since_recent_inq_cat = 3 THEN '1-3 Years Ago'
            WHEN mths_since_recent_inq_cat = 4 THEN '<=1 Year'
        END AS mths_since_recent_inq_label,
        mths_since_recent_bc_cat,
        CASE 
            WHEN mths_since_recent_bc_cat = 0 THEN 'Pre-2012 (No Coverage)'
            WHEN mths_since_recent_bc_cat = 1 THEN 'No Record'
            WHEN mths_since_recent_bc_cat = 2 THEN '>=3 Years Ago'
            WHEN mths_since_recent_bc_cat = 3 THEN '1-3 Years Ago'
            WHEN mths_since_recent_bc_cat = 4 THEN '<=1 Year'
        END AS mths_since_recent_bc_label
    FROM (
        SELECT DISTINCT
            sub_grade, term,
            mths_since_recent_inq_cat, mths_since_recent_bc_cat
        FROM bi_data
    )
""")

print("dim_risk ")
print(con.execute("SELECT COUNT(*) FROM dim_risk").fetchone())

dim_risk 
(904,)


In [26]:
# ── dim_purpose ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE dim_purpose AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY purpose) AS purpose_key,
        purpose
    FROM (SELECT DISTINCT purpose FROM bi_data)
""")

print("dim_purpose ")
print(con.execute("SELECT COUNT(*) FROM dim_purpose").fetchone())

dim_purpose 
(14,)


In [27]:
# ── dim_geography ───────────────────────────────────────────
# Create state to fully qualified name mapping for Azure Maps
state_mapping = {
    'AL': 'Alabama, USA', 'AK': 'Alaska, USA', 'AZ': 'Arizona, USA', 'AR': 'Arkansas, USA',
    'CA': 'California, USA', 'CO': 'Colorado, USA', 'CT': 'Connecticut, USA', 'DE': 'Delaware, USA',
    'FL': 'Florida, USA', 'GA': 'Georgia, USA', 'HI': 'Hawaii, USA', 'ID': 'Idaho, USA',
    'IL': 'Illinois, USA', 'IN': 'Indiana, USA', 'IA': 'Iowa, USA', 'KS': 'Kansas, USA',
    'KY': 'Kentucky, USA', 'LA': 'Louisiana, USA', 'ME': 'Maine, USA', 'MD': 'Maryland, USA',
    'MA': 'Massachusetts, USA', 'MI': 'Michigan, USA', 'MN': 'Minnesota, USA', 'MS': 'Mississippi, USA',
    'MO': 'Missouri, USA', 'MT': 'Montana, USA', 'NE': 'Nebraska, USA', 'NV': 'Nevada, USA',
    'NH': 'New Hampshire, USA', 'NJ': 'New Jersey, USA', 'NM': 'New Mexico, USA', 'NY': 'New York, USA',
    'NC': 'North Carolina, USA', 'ND': 'North Dakota, USA', 'OH': 'Ohio, USA', 'OK': 'Oklahoma, USA',
    'OR': 'Oregon, USA', 'PA': 'Pennsylvania, USA', 'RI': 'Rhode Island, USA', 'SC': 'South Carolina, USA',
    'SD': 'South Dakota, USA', 'TN': 'Tennessee, USA', 'TX': 'Texas, USA', 'UT': 'Utah, USA',
    'VT': 'Vermont, USA', 'VA': 'Virginia, USA', 'WA': 'Washington, USA', 'WV': 'West Virginia, USA',
    'WI': 'Wisconsin, USA', 'WY': 'Wyoming, USA', 'DC': 'District of Columbia, USA'
}
df_states = pd.DataFrame(list(state_mapping.items()), columns=['addr_state', 'state_name_usa'])
con.execute("CREATE OR REPLACE TABLE dim_geography AS SELECT * FROM df_states")

print("dim_geography ")
print(con.execute("SELECT COUNT(*) FROM dim_geography").fetchone())

dim_geography 
(51,)


In [28]:
# ── fact_loans ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE fact_loans AS
    SELECT
        b.loan_id,
        dr.risk_key,
        dp.purpose_key,
        dd.date_key,
        b.addr_state, -- FK to dim_geography
        
        -- Feature Engineering for BI (Calculated Columns)
        CASE
            WHEN b.annual_inc IS NULL THEN 'Unknown'
            WHEN b.annual_inc < 40000 THEN 'Low (<40k)'
            WHEN b.annual_inc >= 40000 AND b.annual_inc < 80000 THEN 'Medium (40k-80k)'
            WHEN b.annual_inc >= 80000 AND b.annual_inc < 120000 THEN 'High (80k-120k)'
            ELSE 'Very High (>120k)'
        END AS annual_inc_category,
        
        -- All other analytical columns (excluding those already selected or normalized into dimensions)
        COLUMNS(c -> c NOT IN ('issue_d', 'sub_grade', 'term', 'mths_since_recent_inq_cat', 'mths_since_recent_bc_cat', 'purpose', 'addr_state', 'loan_id'))
        
    FROM bi_data b
    JOIN dim_risk dr 
        ON b.sub_grade = dr.sub_grade 
        AND b.term = dr.term
        AND b.mths_since_recent_inq_cat = dr.mths_since_recent_inq_cat
        AND b.mths_since_recent_bc_cat = dr.mths_since_recent_bc_cat
    JOIN dim_purpose dp ON b.purpose = dp.purpose
    JOIN dim_date dd ON b.issue_d::DATE = dd.issue_d
""")

print("fact_loans ")
print(con.execute("SELECT COUNT(*) FROM fact_loans").fetchone())

fact_loans 
(2257158,)


In [29]:
# ── Export to Parquet ───────────────────────────────────────
tables = ['fact_loans', 'dim_risk', 'dim_purpose', 'dim_date', 'dim_geography']

for table in tables:
    con.execute(f"""
        COPY {table} TO '../data/gold/{table}.parquet' (FORMAT PARQUET)
    """)
    print(f"{table} exported to parquet ")

print("\nStar Schema complete!")
con.close()


fact_loans exported to parquet 
dim_risk exported to parquet 
dim_purpose exported to parquet 
dim_date exported to parquet 
dim_geography exported to parquet 

Star Schema complete!
